# SARSA
### *Model-free RL, on-policy method*

#### Load the Tic-Tac-Toe environment.

In [3]:
from tic_tac_toe_env import TicTacToe
import random
import numpy as np

In [32]:
def random_move(game: TicTacToe, letter: str):
    board = game.board
    player = letter
    opponent = 'O' if player == 'X' else 'X'

    move = random.choice(game.available_moves())
    
    return move

In [33]:
def choose_move(game: TicTacToe, letter: str):
    # board = list(game.get_flat_state())
    board = game.board
    n = game.n
    player = letter
    opponent = 'O' if player == 'X' else 'X'

    def empty():
        return game.available_moves()

    def lines():
        all_lines = []
        for i in range(n):
            all_lines.append([i*n+j for j in range(n)])
        for j in range(n):
            all_lines.append([i*n+j for i in range(n)])
        # diag
        all_lines.append([i*n+i for i in range(n)])
        # off-diag
        all_lines.append([i*n+(n-1-i) for i in range(n)])
        return all_lines

    def can_win(marker):
        winning_moves = []
        for line in lines():
            vals = [board[i] for i in line]
            if vals.count(marker) == n-1 and vals.count('_') == 1:
                winning_moves.append(line[vals.index('_')])
        return winning_moves # return the list of indicies for winning moves

    # if our wins is not empty then play any of those moves to win (here just pick the first)
    our_wins = can_win(player)
    if our_wins:
        return our_wins[0]

    # if the opponent can win then we just block the move. From the lecture we should just pick it randomly
    opp_wins = can_win(opponent)
    if opp_wins:
        return np.random.choice(opp_wins).item()
    
    empty_cells = empty()

    first_empty = empty_cells[0]
    
    # play sequentially in the row first
    current_row = first_empty//n # recall that these are flattened indices
    next_in_row = first_empty + 1 # next cell in the same row
    
    # if its truly on the same row and empty then play it, otherwise it might wrap around and not make snese
    if next_in_row < (current_row + 1) * n and next_in_row in empty_cells:
        return next_in_row
    
    # now, if thats the case, then try the cell below
    cell_below = first_empty + n
    if cell_below < n*n and cell_below in empty_cells: # so if its actually valid (which it should be) and its empty then play it
        return cell_below
    
    # else play randomly
    return np.random.choice(empty_cells).item()

#### Setup the SARSA Agent.

In [118]:
import random
import copy

class SARSAAgent:
    def __init__(self, alpha=0.5, gamma=0.9, epsilon=0.1, symbol='X'):
        self.Q = {}  # Q-values: {(state_str, action): value}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.symbol = symbol

    def get_state_key(self, board):
        return ''.join(board)

    def choose_action(self, game):
        state_key = self.get_state_key(game.board)
        available = game.available_moves()

        if random.random() < self.epsilon:
            return random.choice(available)  # Explore

        # Exploit
        q_vals = [self.Q.get((state_key, a), 0) for a in available]
        max_q = max(q_vals)
        max_actions = [a for a, q in zip(available, q_vals) if q == max_q]
        return random.choice(max_actions)

    def update(self, s, a, r, s_next, a_next):
        sa = (s, a)
        sa_next = (s_next, a_next)
        q_sa = self.Q.get(sa, 0)
        q_sa_next = self.Q.get(sa_next, 0)
        self.Q[sa] = q_sa + self.alpha * (r + self.gamma * q_sa_next - q_sa)


    def save_q(self, filename="q_values.pkl"):
        import pickle
        with open(filename, "wb") as f:
            pickle.dump(self.Q, f)
        print(f"Q-values saved to '{filename}'")

    def load(self, filename="q_table.pkl"):
        import pickle
        with open(filename, "rb") as f:
            self.q_table = pickle.load(f)

def play_self_play(agent_X, agent_O, board_size, episodes=50000, report_every=1000):

    # Counters for X (the agent we are tracking)
    wins_X = 0
    losses_X = 0
    draws = 0

    # Lists of rates to save results
    win_rates = []
    loss_rates = []
    draw_rates = []

    for ep in range(1, episodes + 1):

        game = TicTacToe(n=board_size)
        state = ''.join(game.board)
        current_agent = agent_X
        next_agent = agent_O
        action = current_agent.choose_action(game)

        while game.empty_squares():
            game.make_move(action, current_agent.symbol)
            reward = 0
            done = game.current_winner is not None or not game.empty_squares()

            if done:
                # -------------------------------
                #   Outcome + statistics update
                # -------------------------------
                if game.current_winner == agent_X.symbol:   # X wins
                    wins_X += 1
                    current_agent.update(state, action, 1, None, None)
                    next_agent.update(state, action, -1, None, None)

                elif game.current_winner == agent_O.symbol: # X loses
                    losses_X += 1
                    current_agent.update(state, action, -1, None, None)
                    next_agent.update(state, action, 1, None, None)

                else:  # draw
                    draws += 1
                    current_agent.update(state, action, 0, None, None)
                    next_agent.update(state, action, 0, None, None)

                break

            # continue playing
            next_state = ''.join(game.board)
            next_action = next_agent.choose_action(game)

            current_agent.update(state, action, 0, next_state, next_action)

            # swap players
            state = next_state
            action = next_action
            current_agent, next_agent = next_agent, current_agent

        # -------------------------
        # Record results every N episodes
        # -------------------------
        if ep % report_every == 0:
            total = wins_X + losses_X + draws
            if total > 0:
                win_rate = wins_X / total
                loss_rate = losses_X / total
                draw_rate = draws / total

                win_rates.append(win_rate)
                loss_rates.append(loss_rate)
                draw_rates.append(draw_rate)

                print(f"Episode {ep}:  Win={win_rate:.3f}  "
                      f"Loss={loss_rate:.3f}  Draw={draw_rate:.3f}")

    # Print lists at the end
    print("\nFinal Win Rates:", win_rates)
    print("Final Loss Rates:", loss_rates)
    print("Final Draw Rates:", draw_rates)

    return win_rates, loss_rates, draw_rates
# def play_self_play(agent_X, agent_O, board_size, episodes=50000):
#     for _ in range(episodes):
#         game = TicTacToe(n=board_size)
#         state = ''.join(game.board)
#         current_agent = agent_X
#         next_agent = agent_O
#         action = current_agent.choose_action(game)

#         while game.empty_squares():
#             game.make_move(action, current_agent.symbol)
#             reward = 0
#             done = game.current_winner is not None or not game.empty_squares()

#             if done:
#                 if game.current_winner == current_agent.symbol:
#                     reward = 1
#                     current_agent.update(state, action, reward, None, None)
#                     next_agent.update(state, action, -1, None, None)
#                 elif game.current_winner == next_agent.symbol:
#                     reward = -1
#                     current_agent.update(state, action, reward, None, None)
#                     next_agent.update(state, action, 1, None, None)
#                 else:
#                     reward = 0
#                     current_agent.update(state, action, reward, None, None)
#                     next_agent.update(state, action, reward, None, None)
#                 break

#             next_state = ''.join(game.board)
#             next_action = next_agent.choose_action(game)

#             current_agent.update(state, action, reward, next_state, next_action)
        
#             # Swap
#             state = next_state
#             action = next_action
#             current_agent, next_agent = next_agent, current_agent


def play_human(agent, board_size, human_symbol='X'):
    game = TicTacToe(n = board_size)
    human_turn = human_symbol == 'X'

    while game.empty_squares():
        game.print_board()
        if human_turn:
            move = int(input("Enter your move (0-8): "))
            if move not in game.available_moves():
                print("Invalid move, try again.")
                continue
            game.make_move(move, human_symbol)
        else:
            move = agent.choose_action(game)
            game.make_move(move, agent.symbol)
            print(f"AI plays: {move}")

        if game.current_winner:
            game.print_board()
            winner = "Human" if human_turn else "AI"
            print(f"{winner} wins!")
            return
        human_turn = not human_turn

    game.print_board()
    print("It's a tie!")




#### Train the SARSA Agent

In [114]:
board_size = 3
num_episodes = 20000
# Create agents
agent_X = SARSAAgent(symbol='X', epsilon=0.1)
agent_O = SARSAAgent(symbol='O', epsilon=0.1)

# Train with self-play
play_self_play(agent_X, agent_O, board_size=board_size, episodes=num_episodes)
# X is always playing first
# Set epsilon to 0 for deterministic play
agent_O.epsilon = 0
agent_X.epsilon = 0


Episode 1000:  Win=0.630  Loss=0.249  Draw=0.121
Episode 2000:  Win=0.649  Loss=0.229  Draw=0.122
Episode 3000:  Win=0.656  Loss=0.215  Draw=0.129
Episode 4000:  Win=0.654  Loss=0.210  Draw=0.137
Episode 5000:  Win=0.644  Loss=0.211  Draw=0.144
Episode 6000:  Win=0.637  Loss=0.214  Draw=0.149
Episode 7000:  Win=0.627  Loss=0.216  Draw=0.157
Episode 8000:  Win=0.610  Loss=0.222  Draw=0.168
Episode 9000:  Win=0.596  Loss=0.228  Draw=0.176
Episode 10000:  Win=0.585  Loss=0.232  Draw=0.183
Episode 11000:  Win=0.576  Loss=0.236  Draw=0.188
Episode 12000:  Win=0.568  Loss=0.239  Draw=0.193
Episode 13000:  Win=0.563  Loss=0.242  Draw=0.195
Episode 14000:  Win=0.557  Loss=0.247  Draw=0.196
Episode 15000:  Win=0.551  Loss=0.251  Draw=0.198
Episode 16000:  Win=0.546  Loss=0.255  Draw=0.199
Episode 17000:  Win=0.541  Loss=0.259  Draw=0.200
Episode 18000:  Win=0.538  Loss=0.261  Draw=0.201
Episode 19000:  Win=0.535  Loss=0.262  Draw=0.203
Episode 20000:  Win=0.532  Loss=0.264  Draw=0.204

Final Wi

In [115]:
agent_X.save_q("dec6_sarsa_q_values_4x4")

Q-values saved to 'dec6_sarsa_q_values_4x4'


In [110]:
def play(agent, n=board_size, print_game=False):
    game = TicTacToe(n=n)

    agent_letter = 'X'
    random_letter = 'O'
    turn = 'O'

    # Use deterministic best moves
    agent.epsilon = 0

    while True:
        if print_game:
            print(game)
            print()

        if turn == agent_letter:
            move = agent.choose_action(game)
            game.make_move(move, agent_letter)
        else:
            move = choose_move(game, random_letter)
            game.make_move(move, random_letter)

        # Check win
        if game.current_winner:
            if turn == agent_letter:
                return 1      # Q-agent wins
            else:
                return -1     # random agent wins

        # Check draw
        if not game.empty_squares():
            return 0

        # Swap turn
        turn = random_letter if turn == agent_letter else agent_letter

In [109]:
def play_ai_first(agent, n=board_size, print_game=False):
    game = TicTacToe(n=n)

    agent_letter = 'X'
    random_letter = 'O'
    turn = 'X'

    # Use deterministic best moves
    agent.epsilon = 0

    while True:
        if print_game:
            print(game)
            print()

        if turn == agent_letter:
            move = agent.choose_action(game)
            game.make_move(move, agent_letter)
        else:
            move = choose_move(game, random_letter)
            game.make_move(move, random_letter)

        # Check win
        if game.current_winner:
            if turn == agent_letter:
                return 1      # Q-agent wins
            else:
                return -1     # random agent wins

        # Check draw
        if not game.empty_squares():
            return 0

        # Swap turn
        turn = random_letter if turn == agent_letter else agent_letter

In [111]:
def evaluate(agent, games=15000, n=board_size):

    wins = 0
    draws = 0
    losses = 0

    ai_wins = []
    ai_draws = []
    ai_loss = []

    for i in range(games):
        result = play(agent, n=n, print_game=False)
        result_ai_first = play_ai_first(agent, n=n, print_game=False)

        # agent goes second
        if result == 1:
            wins += 1
        elif result == 0:
            draws += 1
        else:
            losses += 1

        # agent goes first
        if result_ai_first == 1:
            wins += 1
        elif result_ai_first == 0:
            draws += 1
        else:
            losses += 1

        # Evaluate every 1000 iterations
        if (i + 1) % 1000 == 0:
            total = wins + draws + losses   # always safe, never zero here

            ai_wins.append(wins / total)
            ai_loss.append(losses / total)
            ai_draws.append(draws / total)

    # Print summary
    total_games = games * 2   # because each loop has 2 games
    print(f"Wins:   {wins / total_games}")
    print(f"Draws:  {draws / total_games}")
    print(f"Losses: {losses / total_games}")

    print("Win rates per 1000 games:", ai_wins)
    print("Draw rates per 1000 games:", ai_draws)
    print("Loss rates per 1000 games:", ai_loss)

    return ai_wins, ai_draws, ai_loss


In [112]:
print("board size", board_size)
evaluate(agent_X, games=15000, n=board_size)

board size 10
Wins:   0.014033333333333333
Draws:  0.9726666666666667
Losses: 0.0133
Win rates per 1000 games: [0.0135, 0.0125, 0.013333333333333334, 0.013375, 0.0141, 0.013333333333333334, 0.013714285714285714, 0.013875, 0.013777777777777778, 0.01395, 0.013681818181818182, 0.013833333333333333, 0.013884615384615384, 0.013964285714285714, 0.014033333333333333]
Draw rates per 1000 games: [0.978, 0.97475, 0.9748333333333333, 0.974125, 0.973, 0.9735, 0.9733571428571428, 0.9733125, 0.9735, 0.97315, 0.9735454545454545, 0.9733333333333334, 0.9731153846153846, 0.9729285714285715, 0.9726666666666667]
Loss rates per 1000 games: [0.0085, 0.01275, 0.011833333333333333, 0.0125, 0.0129, 0.013166666666666667, 0.012928571428571428, 0.0128125, 0.012722222222222222, 0.0129, 0.012772727272727272, 0.012833333333333334, 0.013, 0.013107142857142857, 0.0133]


([0.0135,
  0.0125,
  0.013333333333333334,
  0.013375,
  0.0141,
  0.013333333333333334,
  0.013714285714285714,
  0.013875,
  0.013777777777777778,
  0.01395,
  0.013681818181818182,
  0.013833333333333333,
  0.013884615384615384,
  0.013964285714285714,
  0.014033333333333333],
 [0.978,
  0.97475,
  0.9748333333333333,
  0.974125,
  0.973,
  0.9735,
  0.9733571428571428,
  0.9733125,
  0.9735,
  0.97315,
  0.9735454545454545,
  0.9733333333333334,
  0.9731153846153846,
  0.9729285714285715,
  0.9726666666666667],
 [0.0085,
  0.01275,
  0.011833333333333333,
  0.0125,
  0.0129,
  0.013166666666666667,
  0.012928571428571428,
  0.0128125,
  0.012722222222222222,
  0.0129,
  0.012772727272727272,
  0.012833333333333334,
  0.013,
  0.013107142857142857,
  0.0133])

In [ ]:
# Play against human
play_human(agent_X, human_symbol='O')

In [24]:
def play(episode, n):
    print("Episode", episode)

    # declare the tic tac toe environment
    game = TicTacToe(n=n)
    
    # assign player letters
    human_letter = 'X'
    ai_letter = 'O'

    # game.print_board()

    while game.empty_squares():

        # TURN 2: AI
        # the get_move function is where we run the monte carlo simulation
        # ai_move = agent.get_move(game)

        ai_move = agent_O.choose_action(game)
        game.make_move(ai_move, 'O')

        # game.make_move(ai_move, ai_letter)
        # print(f"AI moves at {ai_move}")
        # game.print_board()

        if game.current_winner:
            # print("AI wins!")
            return 1

        if not game.empty_squares():
            # print("It's a tie!")
            return 0
        
                # TURN 1: HUMAN
        move = None
        while move not in game.available_moves():
          move = choose_move(game, "X")
        game.make_move(move, human_letter)
        # game.print_board()

        if game.current_winner:
            # print("Opponent wins!")
            return -1

        if not game.empty_squares():
            # print("It's a tie!")
            return 0

In [66]:
numAIWins = 0
numOppWins = 0

print("AI is always making the first move.")
for i in range(1500): 
  # returns 0 for tie, 1 for AI win, -1 for AI lose
  result = play(i, n=3)
  if result == 1:
    numAIWins += 1
  if result == -1:
    numOppWins += 1

print("Number of AI Wins: ", numAIWins/1500)
print("Number of Opp Wins: ", numOppWins/1500)

AI is always making the first move.
Episode 0
Episode 1
Episode 2
Episode 3
Episode 4
Episode 5
Episode 6
Episode 7
Episode 8
Episode 9
Episode 10
Episode 11
Episode 12
Episode 13
Episode 14
Episode 15
Episode 16
Episode 17
Episode 18
Episode 19
Episode 20
Episode 21
Episode 22
Episode 23
Episode 24
Episode 25
Episode 26
Episode 27
Episode 28
Episode 29
Episode 30
Episode 31
Episode 32
Episode 33
Episode 34
Episode 35
Episode 36
Episode 37
Episode 38
Episode 39
Episode 40
Episode 41
Episode 42
Episode 43
Episode 44
Episode 45
Episode 46
Episode 47
Episode 48
Episode 49
Episode 50
Episode 51
Episode 52
Episode 53
Episode 54
Episode 55
Episode 56
Episode 57
Episode 58
Episode 59
Episode 60
Episode 61
Episode 62
Episode 63
Episode 64
Episode 65
Episode 66
Episode 67
Episode 68
Episode 69
Episode 70
Episode 71
Episode 72
Episode 73
Episode 74
Episode 75
Episode 76
Episode 77
Episode 78
Episode 79
Episode 80
Episode 81
Episode 82
Episode 83
Episode 84
Episode 85
Episode 86
Episode 87
Episod

## playing against human

In [121]:
def play():
    print("Welcome to Tic Tac Toe! You are O. AI is X.")

    # declare the tic tac toe environment
    game = TicTacToe(n=4)
    
    # assign player letters
    human_letter = 'O'
    ai_letter = 'X'

    game.print_board()

    while game.empty_squares():
        # TURN 1: HUMAN
        move = None
        while move not in game.available_moves():
            try:
                move = int(input("Enter your move (0-b): "))
            except ValueError:
                continue
        game.make_move(move, human_letter)
        game.print_board()

        if game.current_winner:
            print("You win!")
            return

        if not game.empty_squares():
            print("It's a tie!")
            return

        # TURN 2: AI
        # the get_move function is where we run the monte carlo simulation
        ai_move = ai.choose_action(game)

        game.make_move(ai_move, ai_letter)
        print(f"AI moves at {ai_move}")
        game.print_board()

        if game.current_winner:
            print("AI wins!")
            return

        if not game.empty_squares():
            print("It's a tie!")
            return

In [122]:
ai = SARSAAgent()
ai.load("dec6_sarsa_q_values_4x4")

In [123]:
play()

Welcome to Tic Tac Toe! You are O. AI is X.
|   |   |   |   |
|   |   |   |   |
|   |   |   |   |
|   |   |   |   |
| O |   |   |   |
|   |   |   |   |
|   |   |   |   |
|   |   |   |   |
AI moves at 2
| O |   | X |   |
|   |   |   |   |
|   |   |   |   |
|   |   |   |   |
| O | O | X |   |
|   |   |   |   |
|   |   |   |   |
|   |   |   |   |
AI moves at 5
| O | O | X |   |
|   | X |   |   |
|   |   |   |   |
|   |   |   |   |
| O | O | X | O |
|   | X |   |   |
|   |   |   |   |
|   |   |   |   |
AI moves at 15
| O | O | X | O |
|   | X |   |   |
|   |   |   |   |
|   |   |   | X |
| O | O | X | O |
| O | X |   |   |
|   |   |   |   |
|   |   |   | X |
AI moves at 6
| O | O | X | O |
| O | X | X |   |
|   |   |   |   |
|   |   |   | X |
| O | O | X | O |
| O | X | X |   |
|   |   |   |   |
| O |   |   | X |
AI moves at 13
| O | O | X | O |
| O | X | X |   |
|   |   |   |   |
| O | X |   | X |


KeyboardInterrupt: Interrupted by user